# 2. Process ASHD and AEDP 148-Household PyNNLF Output

This notebook builds result tables for the new ASHD 148-household weather experiment and combines the ASHD 1-day result with the existing AEDP 148-household 1-day result from `ch5_data.xlsx`.

Run this after `1_run_pynnlf_ashd_148hh.ipynb` has completed the ASHD experiments.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd


def find_publication_project(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "specs" / "ashd_148hh_batch.yaml").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


PROJECT_DIR = find_publication_project(Path.cwd().resolve())
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RECAP_PATH = PROJECT_DIR / "experiment_result" / "a1_experiment_result.csv"
CH5_DATA_PATH = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\05_Thesis\02_Draft\6. Chapter 6. PyNNLF Extension\ch5_data.xlsx")

MODEL_ORDER = [
    "m1_naive_hp1",
    "m2_snaive_hp2",
    "m3_ets_hp1",
    "m4_arima_hp1",
    "m6_lr_hp1",
    "m7_ann_hp1",
    "m8_dnn_hp1",
    "m9_rt_hp3",
    "m10_rf_hp1",
    "m13_lstm_hp2",
    "m16_prophet_hp1",
    "m17_xgb_hp1",
]
HORIZON_ORDER = [30, 1440, 10080]
HORIZON_LABELS = {30: "30_min", 1440: "1_day", 10080: "1_week"}

print(f"Publication project: {PROJECT_DIR}")
print(f"Results directory: {RESULTS_DIR}")


## 2. Load ASHD recap rows

This checks that all 36 ASHD runs exist before building result tables.

In [ ]:
if not RECAP_PATH.exists():
    raise FileNotFoundError(
        f"Missing PyNNLF recap at {RECAP_PATH}. Run 1_run_pynnlf_ashd_148hh.ipynb first."
    )

recap = pd.read_csv(RECAP_PATH)
required = {"dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"}
missing = required - set(recap.columns)
if missing:
    raise ValueError(f"Recap is missing required columns: {sorted(missing)}")

ashd = recap.loc[
    recap["dataset_no"].eq("ds20") & recap["forecast_horizon_min"].isin(HORIZON_ORDER)
].copy()
ashd["model_name"] = pd.Categorical(ashd["model_name"], categories=MODEL_ORDER, ordered=True)
ashd["horizon_label"] = pd.Categorical(
    ashd["forecast_horizon_min"].map(HORIZON_LABELS),
    categories=[HORIZON_LABELS[h] for h in HORIZON_ORDER],
    ordered=True,
)
ashd["dataset_label"] = "ASHD_148hh_weather"
ashd["result_source"] = "publication_journal_article_1_experiment_result"

expected_keys = {(horizon, model) for horizon in HORIZON_ORDER for model in MODEL_ORDER}
actual_keys = set(zip(ashd["forecast_horizon_min"].astype(int), ashd["model_name"].astype(str)))
missing_keys = sorted(expected_keys - actual_keys)
if missing_keys:
    raise ValueError(
        "ASHD ds20 results are incomplete. Run 1_run_pynnlf_ashd_148hh.ipynb first. "
        f"Missing {len(missing_keys)} combinations: {missing_keys[:10]}"
    )
if ashd.shape[0] != len(expected_keys):
    raise ValueError(f"Expected {len(expected_keys)} ASHD rows, found {ashd.shape[0]}")

ashd = ashd.sort_values(["forecast_horizon_min", "model_name"])
display(ashd[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])


## 3. Import existing AEDP 1-day rows

The current Excel result contains AEDP `ds11` rows for the 1-day horizon only. These rows are used for the shared ASHD-vs-AEDP comparison.

In [ ]:
if not CH5_DATA_PATH.exists():
    raise FileNotFoundError(CH5_DATA_PATH)

aedp_sheet = pd.read_excel(CH5_DATA_PATH, sheet_name="3.3.1 dataset")
aedp = aedp_sheet.loc[
    aedp_sheet["dataset_no"].astype(str).eq("ds11")
    & aedp_sheet["forecast_horizon_min"].astype(int).eq(1440)
].copy()

if aedp.empty:
    raise ValueError("Could not find AEDP ds11 1-day rows in ch5_data.xlsx sheet '3.3.1 dataset'.")

aedp["model_name"] = pd.Categorical(aedp["model_name"], categories=MODEL_ORDER, ordered=True)
aedp["horizon_label"] = "1_day"
aedp["dataset_label"] = "AEDP_148hh_weather"
aedp["result_source"] = "ch5_data_xlsx_3.3.1_dataset"
aedp = aedp.sort_values("model_name")

missing_aedp_models = sorted(set(MODEL_ORDER) - set(aedp["model_name"].astype(str)))
if missing_aedp_models:
    raise ValueError(f"AEDP ds11 1-day result is missing models: {missing_aedp_models}")
if aedp.shape[0] != len(MODEL_ORDER):
    raise ValueError(f"Expected {len(MODEL_ORDER)} AEDP rows, found {aedp.shape[0]}")

display(aedp[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])


## 4. Build result tables

In [ ]:
def wide_metric_by_horizon(df: pd.DataFrame, metric_column: str) -> pd.DataFrame:
    table = df.pivot_table(
        index="model_name",
        columns="horizon_label",
        values=metric_column,
        aggfunc="first",
        observed=False,
    )
    table = table.reindex(index=MODEL_ORDER)
    table = table[[HORIZON_LABELS[h] for h in HORIZON_ORDER]]
    table.index.name = "model_hp"
    table.columns.name = None
    return table


def wide_metric_by_dataset(df: pd.DataFrame, metric_column: str) -> pd.DataFrame:
    table = df.pivot_table(
        index="model_name",
        columns="dataset_label",
        values=metric_column,
        aggfunc="first",
        observed=False,
    )
    table = table.reindex(index=MODEL_ORDER)
    table = table[["ASHD_148hh_weather", "AEDP_148hh_weather"]]
    table.index.name = "model_hp"
    table.columns.name = None
    return table


common_columns = [
    "experiment_no",
    "exp_date",
    "dataset_no",
    "dataset_label",
    "forecast_horizon_min",
    "horizon_label",
    "model_no",
    "hyperparameter_no",
    "model_name",
    "test_RMSE",
    "test_RMSE_stddev",
    "test_nRMSE",
    "test_nRMSE_stddev",
    "result_source",
]
for frame in [ashd, aedp]:
    for column in common_columns:
        if column not in frame.columns:
            frame[column] = pd.NA

combined = pd.concat([ashd[common_columns], aedp[common_columns]], ignore_index=True)
combined["model_name"] = pd.Categorical(combined["model_name"], categories=MODEL_ORDER, ordered=True)
combined = combined.sort_values(["dataset_label", "forecast_horizon_min", "model_name"])

ashd_nrmse = wide_metric_by_horizon(ashd, "test_nRMSE")
ashd_stddev = wide_metric_by_horizon(ashd, "test_nRMSE_stddev")
shared_fh8 = combined.loc[combined["forecast_horizon_min"].astype(int).eq(1440)].copy()
comparison_nrmse = wide_metric_by_dataset(shared_fh8, "test_nRMSE")
comparison_stddev = wide_metric_by_dataset(shared_fh8, "test_nRMSE_stddev")

combined.to_csv(RESULTS_DIR / "ashd_aedp_148hh_combined_recap.csv", index=False)
ashd_nrmse.to_csv(RESULTS_DIR / "ashd_148hh_weather_nrmse_by_horizon.csv")
ashd_stddev.to_csv(RESULTS_DIR / "ashd_148hh_weather_nrmse_stddev_by_horizon.csv")
comparison_nrmse.to_csv(RESULTS_DIR / "ashd_aedp_148hh_fh8_nrmse_comparison.csv")
comparison_stddev.to_csv(RESULTS_DIR / "ashd_aedp_148hh_fh8_nrmse_stddev_comparison.csv")

print("Wrote result CSVs to", RESULTS_DIR)
display(ashd_nrmse.round(3))
display(ashd_stddev.round(3))
display(comparison_nrmse.round(3))
display(comparison_stddev.round(3))
